### Initialise

In [1]:
import sqlite3
import pandas as pd
import pyodbc
crm_conn = sqlite3.connect("go_crm_train.sqlite")
sales_conn = sqlite3.connect("go_sales_train.sqlite")  
staff_conn = sqlite3.connect("go_staff_train.sqlite") 


inventory_df = pd.read_csv("inventory_levels_train.csv")
Sales_Staff_df = pd.read_sql("SELECT * FROM sales_staff", staff_conn)
Order_Header_df = pd.read_sql("SELECT * FROM order_header", sales_conn)
Retailer_Site_df = pd.read_sql("SELECT * FROM retailer_site", sales_conn)
Country_df = pd.read_sql("SELECT * FROM country", sales_conn)
Returned_Item_df = pd.read_sql("SELECT * FROM returned_item", sales_conn)
Return_Reason_df = pd.read_sql("SELECT * FROM return_reason", sales_conn)
inventory_df = pd.read_csv("inventory_levels_train.csv")
inventory_levels = inventory_df[['INVENTORY_YEAR', 'INVENTORY_MONTH']].rename(columns={
    'INVENTORY_YEAR': 'YEAR',
    'INVENTORY_MONTH': 'MONTH'
})

product_line_df = pd.read_sql_query("""
        SELECT 
            *
        FROM product_line
""", sales_conn)

product_type_df = pd.read_sql_query("""
        SELECT 
            *
        FROM product_type
    """, sales_conn)

product_df = pd.read_sql_query("""
        SELECT 
            *
        FROM product
    """, sales_conn)

introduction_date = product_df[['INTRODUCTION_DATE']].rename(columns={
    'INTRODUCTION_DATE': 'DATUM'
})

order_header_df = pd.read_sql_query("""
        SELECT 
            *
        FROM order_header
    """, sales_conn)

order_date = order_header_df[['ORDER_DATE']].rename(columns={
    'ORDER_DATE': 'DATUM'
})





### Database legen

In [2]:
import pyodbc

DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse'}

export_conn = pyodbc.connect(
    'DRIVER={SQL Server};SERVER=' +
    DB['servername'] +
    ';DATABASE=' +
    DB['database'] +
    ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()

tables = [
    "MONTH",
    "DATUM",
    "SALES_STAFF",
    "RETAILER_SITE",
    "PRODUCT",
    "FEIT_ORDER",
    "FEIT_RETURN",
    "FEIT_INVENTORY_LEVELS"
]

try:
    export_cursor.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? NOCHECK CONSTRAINT ALL'")
    
    for table in tables:
        export_cursor.execute(f"IF OBJECT_ID('{table}', 'U') IS NOT NULL DELETE FROM {table}")
        print(f"Data verwijderd uit tabel: {table}")
    
    export_cursor.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? CHECK CONSTRAINT ALL'")
    export_conn.commit()
    print("Alle data is succesvol verwijderd. Klaar voor import.")

except Exception as e:
    export_conn.rollback()
    print(f"Fout opgetreden: {e}")
finally:
    print("Verbinding met de database wordt gesloten.")
    # export_cursor.close()
    # export_conn.close()

Data verwijderd uit tabel: MONTH
Data verwijderd uit tabel: DATUM
Data verwijderd uit tabel: SALES_STAFF
Data verwijderd uit tabel: RETAILER_SITE
Data verwijderd uit tabel: PRODUCT
Data verwijderd uit tabel: FEIT_ORDER
Data verwijderd uit tabel: FEIT_RETURN
Data verwijderd uit tabel: FEIT_INVENTORY_LEVELS
Alle data is succesvol verwijderd. Klaar voor import.
Verbinding met de database wordt gesloten.


Datum

In [3]:
import pyodbc

DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse'}

export_conn = pyodbc.connect(
    'DRIVER={SQL Server};SERVER=' +
    DB['servername'] +
    ';DATABASE=' +
    DB['database'] +
    ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()


def move_year():
    """
    Move data from DEDS.inventory_levels and DEDS.product_forecast to STER.YEAR.
    Ensure the DATE value (1st of the month) exists in the DATE table before inserting.
    """
    # Fetch data from DEDS.inventory_levels


    # Combine both dataframes
    df_combined = pd.concat([introduction_date, order_date], ignore_index=True)

    # Drop duplicates
    df_combined = df_combined.drop_duplicates(subset=['DATUM'])

    # Create DATE column (1st of the month)
    # df_combined['DATE'] = pd.to_datetime(df_combined[['YEAR', 'MONTH']].assign(DAY=1)).dt.date

    # Insert data into STER.YEAR
    cursor = export_conn.cursor()
    for index, row in df_combined.iterrows():
        try:
            # Check if the DATE exists in the DATE table
            cursor.execute("SELECT 1 FROM DATE WHERE DATE = ?", (row['DATUM'],))
            if not cursor.fetchone():
                # If the DATUM doesn't exist, insert it into the DATUM table
                cursor.execute("""
                    INSERT INTO DATUM (DATUM)
                    VALUES (?);
                """, (row['DATUM']))

            # # Insert into YEAR table
            # cursor.execute("""
            #     MERGE INTO YEAR AS target
            #     USING (SELECT ? AS YEAR, ? AS MONTH, ? AS DATUM) AS source
            #     ON target.YEAR = source.YEAR AND target.MONTH = source.MONTH
            #     WHEN NOT MATCHED THEN
            #         INSERT (YEAR, MONTH, DATUM)
            #         VALUES (source.YEAR, source.MONTH, source.DATUM);
            # """, (row['YEAR'], row['MONTH'], row['DATUM']))
        except pyodbc.Error as e:
            print(f"Error processing row {index}: {e}")
            print(f"YEAR and MONTH causing error: {row['YEAR']}, {row['MONTH']}")

    # Commit changes
    export_conn.commit()

move_year()

Error processing row 0: ('42S02', "[42S02] [Microsoft][ODBC SQL Server Driver][SQL Server]Invalid object name 'DATE'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC SQL Server Driver][SQL Server]Statement(s) could not be prepared. (8180)")


KeyError: 'YEAR'

sales_staff

In [ ]:
import pandas as pd
from datetime import datetime

DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse'}

export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                           DB['servername'] + 
                           ';DATABASE=' + 
                           DB['database'] +
                           ';Trusted_Connection=yes')
export_cursor = export_conn.cursor()

current_date = datetime.now()

for index, row in Sales_Staff_df.iterrows():
    try:
        # Converteer DATE_HIRED naar datetime object
        hire_date = pd.to_datetime(row['DATE_HIRED'])
        full_name = f"{row['FIRST_NAME']} {row['LAST_NAME']}".strip()
        
        # Bereken maanden in dienst
        months_employed = (current_date.year - hire_date.year) * 12 + (current_date.month - hire_date.month)
        
        query = """
        INSERT INTO SALES_STAFF
        (SALES_STAFF_CODE, FIRST_NAME, LAST_NAME, FULL_NAME, POSITION_EN, WORK_PHONE, 
        EXTENSION, EMAIL, SALES_BRANCH_CODE, MANAGER_CODE, MONTHS_EMPLOYED)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        data = (
            int(row['SALES_STAFF_CODE']),
            str(row['FIRST_NAME']),
            str(row['LAST_NAME']),
            full_name,
            str(row['POSITION_EN']),
            str(row['WORK_PHONE']),
            int(row['EXTENSION']) if pd.notna(row['EXTENSION']) else None,
            str(row['EMAIL']),
            int(row['SALES_BRANCH_CODE']),
            int(row['MANAGER_CODE']),
            int(months_employed)
        )
        export_cursor.execute(query, data)

    except Exception as e:
        print(f"Fout bij rij {index}: {e}\nData: {row}")

export_conn.commit()
export_cursor.close()
export_conn.close()

RETAILER_SITE

In [ ]:
DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse'}

export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                           DB['servername'] + 
                           ';DATABASE=' + 
                           DB['database'] +
                           ';Trusted_Connection=yes')
export_cursor = export_conn.cursor()

merged_df = pd.merge(Retailer_Site_df, Country_df, on='COUNTRY_CODE', how='inner')
print("Beschikbare kolommen:", merged_df.columns.tolist())

def create_place(row):
    """Combineer CITY, REGION en COUNTRY tot PLACE met komma's, sla lege waarden over"""
    parts = []
    if pd.notna(row.get('CITY')) and str(row['CITY']).strip():
        parts.append(str(row['CITY']).strip())
    if pd.notna(row.get('REGION')) and str(row['REGION']).strip():
        parts.append(str(row['REGION']).strip())
    if pd.notna(row.get('COUNTRY')) and str(row['COUNTRY']).strip():
        parts.append(str(row['COUNTRY']).strip())
    return ', '.join(parts) if parts else None

for index, row in merged_df.iterrows():
    try:
        # Genereer PLACE dynamisch
        place = create_place(row)
        
        query = """
        INSERT INTO Retailer_Site
        (RETAILER_SITE_CODE, RETAILER_CODE, ADDRESS2, ADDRESS1, CITY, REGION, 
         COUNTRY_CODE, ACTIVE_INDICATOR, POSTAL_ZONE, COUNTRY, PLACE) 
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        data = (
            int(row['RETAILER_SITE_CODE']) if pd.notna(row['RETAILER_SITE_CODE']) else None,
            int(row['RETAILER_CODE']) if pd.notna(row['RETAILER_CODE']) else None,
            str(row['ADDRESS2']) if pd.notna(row['ADDRESS2']) else None,
            str(row['ADDRESS1']) if pd.notna(row['ADDRESS1']) else None,
            str(row['CITY']) if pd.notna(row['CITY']) else None,
            str(row['REGION']) if pd.notna(row['REGION']) else None,
            str(row['COUNTRY_CODE']) if pd.notna(row['COUNTRY_CODE']) else None,
            int(row['ACTIVE_INDICATOR']) if pd.notna(row['ACTIVE_INDICATOR']) else 0,  # Default 0 voor actief
            str(row['POSTAL_ZONE']) if pd.notna(row['POSTAL_ZONE']) else None,
            str(row['COUNTRY']) if pd.notna(row['COUNTRY']) else None,
            place  # Gebruik de gegenereerde PLACE
        )
        
        export_cursor.execute(query, data)
   
    except Exception as e:
        print(f"\nFout bij rij {index}: {str(e)}\nProbleemdata:")
        print({col: row[col] for col in row.index if pd.notna(row[col])})
        export_conn.rollback()

export_conn.commit()
export_cursor.close()
export_conn.close()

Beschikbare kolommen: ['RETAILER_SITE_CODE', 'RETAILER_CODE', 'ADDRESS1', 'ADDRESS2', 'CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY_CODE', 'ACTIVE_INDICATOR', 'COUNTRY', 'LANGUAGE', 'CURRENCY_NAME']


product

In [ ]:
DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse' }

export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                              DB['servername'] + 
                              ';DATABASE=' + 
                              DB['database'] +
                              ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()

def move_product():


    product_type_with_line = pd.merge(
        product_type_df,
        product_line_df,
        on='PRODUCT_LINE_CODE',
        how='left'
    )
    
    # Daarna het resultaat met product_df joinen
    final_product_df = pd.merge(
        product_df,
        product_type_with_line,
        on='PRODUCT_TYPE_CODE',
        how='left'
    )

     # Selecteer alleen de gewenste kolommen
    product_feit_df = final_product_df[[
        "PRODUCT_NUMBER",
        "INTRODUCTION_DATE",
        "PRODUCTION_COST",
        "MARGIN",
        "PRODUCT_NAME",
        "PRODUCT_LINE_EN",
        "PRODUCT_LINE_CODE",
        "PRODUCT_TYPE_EN",
        "PRODUCT_TYPE_CODE"
    ]]

    for _, row in product_feit_df.iterrows():
        try:
            query = """
            INSERT INTO PRODUCT (
                PRODUCT_NUMBER, INTRODUCTION_DATE, PRODUCTION_COST, MARGIN, PRODUCT_NAME,
                PRODUCT_LINE_EN, PRODUCT_LINE_CODE, PRODUCT_TYPE_EN, PRODUCT_TYPE_CODE
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """
            params = (
                row['PRODUCT_NUMBER'],
                (row['INTRODUCTION_DATE']) if pd.notna(row['INTRODUCTION_DATE']) else None, 
                (row['PRODUCTION_COST']) if pd.notna(row['PRODUCTION_COST']) else None,
                (row['MARGIN']) if pd.notna(row['MARGIN']) else None,
                (row['PRODUCT_NAME']) if pd.notna(row['PRODUCT_NAME']) else None,
                (row['PRODUCT_LINE_EN']) if pd.notna(row['PRODUCT_LINE_EN']) else None,
                (row['PRODUCT_LINE_CODE']) if pd.notna(row['PRODUCT_LINE_CODE']) else None,
                (row['PRODUCT_TYPE_EN']) if pd.notna(row['PRODUCT_TYPE_EN']) else None,
                (row['PRODUCT_TYPE_CODE']) if pd.notna(row['PRODUCT_TYPE_CODE']) else None,
            )
            export_cursor.execute(query, params)
        except pyodbc.Error as e:
            print(f"Fout in Product (PRODUCT_NUMBER: {row['PRODUCT_NUMBER']}): {e}")
            export_conn.rollback()
        except Exception as e:
            print(f"Onverwachte fout in Product (PRODUCT_NUMBER: {row['PRODUCT_NUMBER']}): {str(e)[:200]}")
            export_conn.rollback()
    
    export_conn.commit()
    print(f"Product migratie voltooid. {len(product_feit_df)} rijen verwerkt.")

    export_cursor.close()
    export_conn.close()
move_product()

Product migratie voltooid. 115 rijen verwerkt.


feit_order

In [ ]:
DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse' }


export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                              DB['servername'] + 
                              ';DATABASE=' + 
                              DB['database'] +
                              ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()

def move_order():


    order_method_df = pd.read_sql_query("""
        SELECT 
            *
        FROM order_method
    """, sales_conn)

    order_details_df = pd.read_sql_query("""
        SELECT 
            *
        FROM order_details
    """, sales_conn)

    order_header_with_order_method = pd.merge(
        order_header_df,
        order_method_df,
        on='ORDER_METHOD_CODE',
        how='left'
    )
    
    # Daarna het resultaat met product_df joinen
    final_order_df = pd.merge(
        order_details_df,
        order_header_with_order_method,
        on='ORDER_NUMBER',
        how='left'
    )

     # Selecteer alleen de gewenste kolommen
    order_feit_df = final_order_df[[
        "ORDER_DETAIL_CODE",
        "ORDER_NUMBER",
        "PRODUCT_NUMBER",
        "QUANTITY",
        "UNIT_COST",
        "UNIT_PRICE",
        "UNIT_SALE_PRICE",
        "RETAILER_NAME",
        "RETAILER_SITE_CODE",
        "SALES_STAFF_CODE",
        "ORDER_DATE",
    ]]

    for _, row in order_feit_df.iterrows():
        try:
            unit_sale_price = float(row['UNIT_SALE_PRICE'])
            quantity = int(row['QUANTITY'])
            unit_cost = float(row['UNIT_COST'])
            profit = (unit_sale_price - unit_cost) * quantity 
            revenue = unit_sale_price * quantity
            
            query = """
            INSERT INTO FEIT_ORDER (
            "ORDER_DETAIL_CODE",
            "ORDER_NUMBER",
            "PRODUCT_NUMBER",
            "QUANTITY",
            "UNIT_COST",
            "UNIT_PRICE",
            "UNIT_SALE_PRICE",
            "RETAILER_NAME",
            "RETAILER_SITE_CODE",
            "SALES_STAFF_CODE",
            "ORDER_DATE",
            "PROFIT",
            "REVENUE"
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """
            
            params = (
                row['ORDER_DETAIL_CODE'],
                (row['ORDER_NUMBER']) if pd.notna(row['ORDER_NUMBER']) else None, 
                (row['PRODUCT_NUMBER']) if pd.notna(row['PRODUCT_NUMBER']) else None,
                (row['QUANTITY']) if pd.notna(row['QUANTITY']) else None,
                (row['UNIT_COST']) if pd.notna(row['UNIT_COST']) else None,
                (row['UNIT_PRICE']) if pd.notna(row['UNIT_PRICE']) else None,
                (row['UNIT_SALE_PRICE']) if pd.notna(row['UNIT_SALE_PRICE']) else None,
                (row['RETAILER_NAME']) if pd.notna(row['RETAILER_NAME']) else None,
                (row['RETAILER_SITE_CODE']) if pd.notna(row['RETAILER_SITE_CODE']) else None,
                (row['SALES_STAFF_CODE']) if pd.notna(row['SALES_STAFF_CODE']) else None,
                (row['ORDER_DATE']) if pd.notna(row['ORDER_DATE']) else None,
                profit,
                revenue
            )
            export_cursor.execute(query, params)
        except pyodbc.Error as e:
            print(f"Fout in Product (ORDER_NUMBER: {row['ORDER_NUMBER']}): {e}")
            export_conn.rollback()
        except Exception as e:
            print(f"Onverwachte fout in Product (ORDER_NUMBER: {row['ORDER_NUMBER']}): {str(e)[:200]}")
            export_conn.rollback()
    
    export_conn.commit()
    print(f"Product migratie voltooid. {len(order_feit_df)} rijen verwerkt.")

    export_cursor.close()
    export_conn.close()
move_order()

Product migratie voltooid. 37757 rijen verwerkt.


Feit_return

In [ ]:
import pandas as pd
from datetime import datetime

DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse'}

export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                           DB['servername'] + 
                           ';DATABASE=' + 
                           DB['database'] +
                           ';Trusted_Connection=yes')
export_cursor = export_conn.cursor()

merged_df = pd.merge(Returned_Item_df, Return_Reason_df, on='RETURN_REASON_CODE', how='inner')
print("Beschikbare kolommen:", merged_df.columns.tolist())

# Eerst berekenen we het jaarlijks retourbedrag voor alle records
if 'RETURN_DATE' in merged_df.columns and 'RETURN_QUANTITY' in merged_df.columns:
    # Converteer RETURN_DATE naar datetime
    merged_df['RETURN_DATE'] = pd.to_datetime(merged_df['RETURN_DATE'])
    
    # Bereken YEARLY_RETURN_AMOUNT per product/klant (pas dit aan naar je businesslogica)
    # Voorbeeld 1: Eenvoudige telling per jaar
    yearly_counts = merged_df.groupby(merged_df['RETURN_DATE'].dt.year)['RETURN_QUANTITY'].sum()
    
    # Voeg berekende waarden toe aan dataframe
    merged_df['YEARLY_RETURN_AMOUNT'] = merged_df['RETURN_DATE'].dt.year.map(yearly_counts)
    
    # OF Voorbeeld 2: Gebaseerd op RETURN_QUANTITY
    # merged_df['YEARLY_RETURN_AMOUNT'] = merged_df['RETURN_QUANTITY'] * 12  # Simpele schatting

for index, row in merged_df.iterrows():
    try:
        # Als YEARLY_RETURN_AMOUNT niet berekend kon worden, gebruik dan RETURN_QUANTITY als fallback
        yearly_return = row.get('YEARLY_RETURN_AMOUNT', row['RETURN_QUANTITY'])
        
        query = """
        INSERT INTO FEIT_RETURN
        (RETURN_CODE, RETURN_DATE, ORDER_DETAIL_CODE, RETURN_REASON_CODE, RETURN_QUANTITY,
        RETURN_DESCRIPTION_EN, YEARLY_RETURN_AMOUNT) 
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """
        
        data = (
            int(row['RETURN_CODE']) if pd.notna(row['RETURN_CODE']) else None,
            row['RETURN_DATE'].strftime('%Y-%m-%d') if pd.notna(row['RETURN_DATE']) else None,
            int(row['ORDER_DETAIL_CODE']) if pd.notna(row['ORDER_DETAIL_CODE']) else None,
            int(row['RETURN_REASON_CODE']) if pd.notna(row['RETURN_REASON_CODE']) else None,
            int(row['RETURN_QUANTITY']),
            str(row['RETURN_DESCRIPTION_EN']),
            int(yearly_return)  # Gebruik de berekende waarde
        )
        
        export_cursor.execute(query, data)
   
    except Exception as e:
        print(f"\nFout bij rij {index}: {str(e)}\nProbleemdata:")
        print({col: row[col] for col in row.index if pd.notna(row[col])})
        export_conn.rollback()

export_conn.commit()
export_cursor.close()
export_conn.close()

Beschikbare kolommen: ['RETURN_CODE', 'RETURN_DATE', 'ORDER_DETAIL_CODE', 'RETURN_REASON_CODE', 'RETURN_QUANTITY', 'RETURN_DESCRIPTION_EN']


FEIT_INVENTORY_LEVELS

In [ ]:
DB = {'servername': r'LAPTOP-VAN-GIJO\SQLEXPRESS', 'database': 'DataWareHouse' }

export_conn = pyodbc.connect('DRIVER={SQL Server};SERVER=' +
                              DB['servername'] + 
                              ';DATABASE=' + 
                              DB['database'] +
                              ';Trusted_Connection=yes'
)

export_cursor = export_conn.cursor()

for index, row in inventory_df.iterrows():
    try:
        query = """
    INSERT INTO FEIT_INVENTORY_LEVELS
    (PRODUCT_NUMBER, INVENTORY_YEAR, INVENTORY_MONTH, INVENTORY_COUNT) 
    VALUES (?, ?, ?, ?)
    """
        data = (
            int(row['PRODUCT_NUMBER']),
            int(row['INVENTORY_YEAR']),
            int(row['INVENTORY_MONTH']),
            int(row['INVENTORY_COUNT'])
        )
        export_cursor.execute(query, data)
    except Exception as e:
        print(f"Fout bij rij {index}: {e}\nData: {row}")

export_conn.commit()
export_cursor.close()
export_conn.close()


Fout bij rij 0: ('23000', '[23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__FEIT_INVE__INVEN__4AB81AF0". The conflict occurred in database "DataWareHouse", table "dbo.MONTH", column \'INVENTORY_MONTH\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The statement has been terminated. (3621)')
Data: INVENTORY_CODE        0
INVENTORY_YEAR     2023
INVENTORY_MONTH       4
PRODUCT_NUMBER       48
INVENTORY_COUNT    1932
Name: 0, dtype: int64
Fout bij rij 1: ('23000', '[23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__FEIT_INVE__INVEN__4AB81AF0". The conflict occurred in database "DataWareHouse", table "dbo.MONTH", column \'INVENTORY_MONTH\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The statement has been terminated. (3621)')
Data: INVENTORY_CODE        1
INVENTORY_YEAR     2

Fout bij rij 22: ('23000', '[23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__FEIT_INVE__INVEN__4AB81AF0". The conflict occurred in database "DataWareHouse", table "dbo.MONTH", column \'INVENTORY_MONTH\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The statement has been terminated. (3621)')
Data: INVENTORY_CODE       22
INVENTORY_YEAR     2023
INVENTORY_MONTH       4
PRODUCT_NUMBER       70
INVENTORY_COUNT     803
Name: 22, dtype: int64
Fout bij rij 23: ('23000', '[23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__FEIT_INVE__INVEN__4AB81AF0". The conflict occurred in database "DataWareHouse", table "dbo.MONTH", column \'INVENTORY_MONTH\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC SQL Server Driver][SQL Server]The statement has been terminated. (3621)')
Data: INVENTORY_CODE       23
INVENTORY_YEAR   